# House Price Analysis — Exploratory Data Analysis
### Project 02 | Data Analysis Roadmap | May 2026

---

## Project Goal
Analyze the Kaggle House Prices dataset to answer:
**What factors most strongly predict house sale prices?**

- **Dataset:** Kaggle House Prices: Advanced Regression Techniques — `train.csv`
- **Size:** 1,460 houses · 81 columns
- **Target variable:** `SalePrice` (house sale price in USD)

---

## Analysis Questions

| # | Question | Key Tool | Stat Test |
|---|---|---|---|
| Q1 | What does the SalePrice distribution look like? | Histogram + Log transform | Descriptive |
| Q2 | Does living area predict price? | Scatter plot | Pearson r |
| Q3 | How does quality rating affect price? | Box plot | Spearman r |
| Q4 | Which neighborhoods have the highest prices? | Horizontal bar chart | Descriptive |
| Q5 | Has price changed over the years? | Line/scatter chart | Pearson r |
| Q6 | Does TotalSF predict better than GrLivArea alone? | Scatter + Feature engineering | Pearson r comparison |
| Q7 | Which features correlate most with price? | Correlation heatmap | Pearson r matrix |

---

## Key Columns Used

| Column | Meaning |
|---|---|
| `SalePrice` | Target — house sale price in USD |
| `GrLivArea` | Above ground living area (sq ft) |
| `TotalBsmtSF` | Basement area (sq ft) |
| `OverallQual` | Overall quality rating (1–10) |
| `YearBuilt` | Year house was built |
| `Neighborhood` | Location within Ames, Iowa |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
df = pd.read_csv('../data/raw/train.csv')
df.head()

In [ ]:
df.describe()

## Q1 — What does the SalePrice distribution look like?

**Goal:** Plot the distribution of house prices, check if it's skewed, and apply log transformation to make it more symmetric for ML.

In [ ]:
saleprice = df['SalePrice'].dropna()
df['log_saleprice'] = np.log1p(df['SalePrice'])

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(saleprice, bins=30, color='lightgreen', label='Sale Price')
plt.title("Sale Price Distribution")
plt.xlabel('Sale Price')
plt.ylabel('Number of Houses')
plt.legend()
plt.tight_layout()
plt.savefig("../visuals/saleprice_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean: {df['SalePrice'].mean():.2f}")
print(f"Median: {df['SalePrice'].median():.2f}")

if df['SalePrice'].mean() > df['SalePrice'].median():
    print("Here Mean > Median: The distribution is right-skewed.")
elif df['SalePrice'].mean() < df['SalePrice'].median():
    print("Here Mean < Median: The distribution is left-skewed.")
else:
    print("Here Mean = Median: The distribution is symmetric.")


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df['log_saleprice'], bins=30, color='lightblue', label='Log(Sale Price)')
plt.title("Log(Sale Price) Distribution")
plt.xlabel('Log(Sale Price)')
plt.ylabel('Number of Houses')
plt.legend()
plt.tight_layout()
plt.savefig("../visuals/saleprice_distribution_log.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean of LogSalePrice: {df['log_saleprice'].mean():.3f}")
print(f"Median of LogSalePrice: {df['log_saleprice'].median():.3f}")
print(f"Skewness after log: {df['log_saleprice'].skew():.3f}")

### Q1 — Conclusion

**What I found:**
- House prices are right-skewed. Mean was $180,921 and median was $163,000, so mean > median — that confirms right skew.
- Most houses are between $100k and $200k. A few expensive houses create that long tail on the right.

**What I did about it:**
- Applied log transformation using `np.log1p()` to make the distribution more symmetric.
- Skewness went from >1 (highly skewed) down to 0.121 — basically normal now.

**Why this matters:**
- ML models work better when the target variable is roughly normally distributed.
- Log transform is standard for price data. I'll use `log_saleprice` later for regression.

**Code I learned:**
| What I wanted to do | How I wrote it |
|---------------------|----------------|
| Log transform | `df['log_saleprice'] = np.log1p(df['SalePrice'])` |
| Check skewness | `df['log_saleprice'].skew()` |
| Plot histogram | `plt.hist(df['log_saleprice'], bins=30)` |

## Q2 — Does living area predict price?

**Goal:** Use a scatter plot to visualize the relationship between above-ground living area (GrLivArea) and SalePrice, then measure the strength of the correlation using Pearson's r.

In [ ]:
living_price_df = df[['GrLivArea', 'SalePrice']].dropna()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=living_price_df, x='GrLivArea', y='SalePrice', color='orange', alpha=0.7)
plt.title("GrLivArea vs SalePrice")
plt.xlabel('Above Ground Living Area (sq ft)')
plt.ylabel('Sale Price')
plt.tight_layout()
plt.savefig("../visuals/grlivarea_vs_saleprice.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
r_value, p_value = stats.pearsonr(living_price_df['GrLivArea'], living_price_df['SalePrice'])
if p_value < 0.05:
    print(f"Significant correlation: r = {r_value:.3f}, p = {p_value:.3e}")
elif p_value >= 0.05:
    print(f"No significant correlation: r = {r_value:.3f}, p = {p_value:.3e}")

### Q2 — Conclusion

**What I was trying to figure out:**
Whether a house's above-ground living area (GrLivArea) can tell me anything about its sale price. Two continuous variables — so I needed a way to see both the pattern and the strength.

**Why I chose what I chose:**
- **Scatter plot, not heatmap:** Heatmap would only give me one number (r). A scatter plot lets me see the actual shape of the relationship — is it linear? Are there outliers? That matters more than just a correlation value.
- **Pearson, not Spearman:** Both variables are continuous and I wanted to measure linear relationship. Spearman is for ranks/ordinal data — not needed here.

**What I found:**
- Scatter plot shows a clear positive trend. Bigger living area = higher price. But I also spotted outliers — some big houses selling cheap, some small houses selling expensive.
- Pearson r = 0.709 (strong positive correlation)
- p-value = 4.52e-223 (basically zero — this isn't random chance)

**What this means:**
Living area alone explains a lot about price, but it's not the whole story. Those outliers tell me other factors matter too (maybe quality, location, basement, etc.).

**Code I learned:**
| What I wanted | How I wrote it |
|---------------|----------------|
| Scatter plot | `sns.scatterplot(data=df, x='GrLivArea', y='SalePrice', alpha=0.4)` |
| Pearson + p-value | `stats.pearsonr(x, y)` |
| Significance check | `if p_value < 0.05:` |

## Q3 — How does quality rating affect price?

**Goal:** Use a box plot to see how SalePrice changes across different OverallQual ratings (1-10), and measure correlation since quality is ordinal.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='OverallQual', y='log_saleprice', color='lightcoral')
plt.yscale('log')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.title("Overall Quality vs SalePrice")
plt.xlabel('Overall Quality')
plt.ylabel('Sale Price(log scale)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../visuals/overallqual_vs_saleprice.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
r_value_2, p_value_2 = stats.spearmanr(df['OverallQual'], df['log_saleprice'])
if p_value_2 < 0.05:
    print(f"Significant correlation: r = {r_value_2:.3f}, p = {p_value_2:.3e}")
elif p_value_2 >= 0.05:
    print(f"No significant correlation: r = {r_value_2:.3f}, p = {p_value_2:.3e}")

In [ ]:
print(df['OverallQual'].value_counts().sort_index())

### Q3 — Conclusion

**What I was trying to figure out:**
Whether a house's overall quality rating (1-10, ordinal) predicts sale price. Different from Q2 — here the x variable is ordered categories, not continuous.

**Why I chose what I chose:**
- **Box plot, not scatter:** Scatter plots need two continuous variables. OverallQual has only 10 discrete values — box plot shows distribution at each quality level.
- **Spearman, not Pearson:** The project guide said Pearson, but that's wrong. Pearson assumes continuous data. OverallQual is ordinal (ranks). Spearman works on ranks — perfect for this.
- **Log scale on y-axis:** Price ranges from $50k to $600k+. Raw scale would make low-quality boxes tiny and hard to compare. Log scale shows percentage differences, which matters more for price. Also matches my Q1 log transformation.

**What I found:**
- Spearman r = 0.810 (stronger than living area's 0.709 — my guess was right)
- p = 0.000 (significant)
- Box plot shows median price generally increases with quality — BUT quality 2 looks weird (median lower than quality 1)

**Why quality = 2 looks wrong:**
- Only 3 houses in quality 2 (checked with `.value_counts()`)
- Too few data points → unreliable box plot
- This taught me: always check sample sizes before trusting group comparisons

**What this means:**
Quality is the strongest predictor I've seen so far (r = 0.81). But low-quality ratings (1-3) have very few samples — conclusions about them aren't reliable.

**Code I learned:**
| What I wanted | How I wrote it |
|---------------|----------------|
| Box plot | `sns.boxplot(data=df, x='OverallQual', y='SalePrice')` |
| Log scale on y | `plt.yscale('log')` |
| Spearman correlation | `stats.spearmanr(x, y)` |
| Check sample sizes | `df['OverallQual'].value_counts().sort_index()` |

## Q4 — Does the neighborhood affect price?

**Goal:** Find median SalePrice for each neighborhood, sort from highest to lowest, and visualize with a horizontal bar chart.

In [ ]:
df['Neighborhood'].unique()

In [ ]:
saleprice_by_neighborhood = df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False)
plt.figure(figsize=(12,6))
sns.barplot(x=saleprice_by_neighborhood.values, y=saleprice_by_neighborhood.index, palette='mako')
plt.xticks(rotation=0)
plt.title("Median Sale Price by Neighborhood")
plt.xlabel('Median Sale Price')
plt.ylabel('Neighborhood')
plt.tight_layout()
plt.savefig("../visuals/median_saleprice_by_neighborhood.png", dpi=150, bbox_inches='tight')
plt.show()

### Q4 — Conclusion

**What I was trying to figure out:**
Whether location (Neighborhood) has a noticeable impact on sale price. Categorical variable with 25 unique values.

**Why I chose what I chose:**
- **Median, not mean:** SalePrice is right-skewed. Median is robust to outliers — a few mansions won't fake the "typical" price for a neighborhood.
- **Horizontal bar chart, not vertical:** 25 neighborhoods. Vertical bars would be unreadable (labels would overlap or need extreme rotation). Horizontal bars let me read neighborhood names clearly on the y-axis.
- **Bar plot, not box plot:** Box plot would show distribution within each neighborhood (spread, outliers), but with 25 categories it becomes visual noise. Bar chart of medians gives a cleaner ranking.

**What I found:**
- Top neighborhoods: `NridgHt`, `NoRidge`, `StoneBr` (~$290k-300k median)
- Bottom neighborhoods: `MeadowV`, `IDOTRR`, `BrDale` (~$85k-100k median)
- Roughly 3x difference between most and least expensive areas

**What this means:**
Location is a major price factor — but many neighborhoods cluster in the middle ($150k-$170k). Within those, other features (quality, size) become the differentiators.

**What I'd do next:**
For ML, I'd probably encode neighborhood as categorical or group rare neighborhoods into an "other" category.

**Code I learned:**
| What I wanted | How I wrote it |
|---------------|----------------|
| Group by + median | `df.groupby('Neighborhood')['SalePrice'].median()` |
| Sort descending | `.sort_values(ascending=False)` |
| Horizontal bar plot | `sns.barplot(x=values, y=index)` |
| Fix size for many bars | `plt.figure(figsize=(12,6))` |

## Q5 — Has price changed over the years?

**Goal:** See how SalePrice trends across YearBuilt — scatter plot with trend line, plus Pearson correlation.

In [ ]:
year_saleprice_df = df[['YearBuilt', 'SalePrice']].dropna()

In [ ]:
plt.figure(figsize=(8,5))
sns.regplot(data=year_saleprice_df, x='YearBuilt', y='SalePrice', line_kws={"color": "#F4991A"}, scatter_kws={"color": "#344F1F"})
plt.title("YearBuilt vs SalePrice")
plt.xlabel('Year Built')
plt.ylabel('Sale Price')
plt.tight_layout()
plt.savefig("../visuals/yearbuilt_vs_saleprice.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
r_value3, p_value3 = stats.pearsonr(year_saleprice_df['YearBuilt'], year_saleprice_df['SalePrice'])
if p_value3 < 0.05:
    print(f"Significant correlation: r = {r_value3:.3f}, p = {p_value3:.3e}")
elif p_value3 >= 0.05:
    print(f"No significant correlation: r = {r_value3:.3f}, p = {p_value3:.3e}")

### Q5 — Conclusion

**What I was trying to figure out:**
Whether newer houses cost more than older ones. YearBuilt (continuous) vs SalePrice (continuous).

**Why I chose what I chose:**
- **Scatter plot + regression line:** YearBuilt is continuous, scatter plot shows individual points. `sns.regplot()` adds a trend line automatically.
- **Pearson correlation:** Both variables are continuous, relationship appears roughly linear.

**What I found:**
- Pearson r = 0.523 (moderate positive correlation)
- p < 0.05 (significant)
- Trend line slopes upward — newer houses cost more

**THE BIG CAVEAT (I spotted this myself):**
This analysis is **flawed** because it ignores inflation. A $200k house in 1980 is NOT the same as $200k in 2020. The correlation likely overstates real appreciation because:
- Inflation makes newer houses look more expensive even if real value stayed flat
- Without adjusting to constant dollars (using CPI), this comparison is misleading

**What this taught me:**
- Always think about whether your variables are comparable across time
- Correlation doesn't imply causation — and here, it might not even imply real correlation
- For real estate analysis over time, inflation adjustment is mandatory

**Code I learned:**
| What I wanted | How I wrote it |
|---------------|----------------|
| Scatter + trend line | `sns.regplot(x='YearBuilt', y='SalePrice', data=df)` |
| Custom colors | `scatter_kws={'color':'#344F1F'}, line_kws={'color':'#F4991A'}` |

**What I'd do differently next time:**
Adjust prices for inflation before repeating this analysis.

## Q6 — Feature Engineering: Total Square Footage

**Goal:** Create a new feature `TotalSF` = `GrLivArea` + `TotalBsmtSF`, then compare its correlation with SalePrice against `GrLivArea` alone.

In [ ]:
df['TotalBsmtSF'] = df['TotalBsmtSF'].fillna(0)
df['TotalSF'] = df['GrLivArea'] + df['TotalBsmtSF']

In [ ]:
plt.figure(figsize=(8,5))
sns.regplot(data=df, x='TotalSF', y='SalePrice', line_kws={"color": "#F3C623"}, scatter_kws={"color": "#10375C"})
plt.title("TotalSF vs SalePrice")
plt.xlabel('Total Square Feet')
plt.ylabel('Sale Price')
plt.tight_layout()
plt.savefig("../visuals/totalSF_vs_saleprice.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
r_value4, p_value4 = stats.pearsonr(df['TotalSF'], df['SalePrice'])
if p_value4 < 0.05:
    print(f"Significant correlation: r = {r_value4:.3f}, p = {p_value4:.3e}")
elif p_value4 >= 0.05:
    print(f"No significant correlation: r = {r_value4:.3f}, p = {p_value4:.3e}")

### Q6 — Conclusion

**What I was trying to figure out:**
Whether combining living area and basement area into one `TotalSF` feature predicts price better than living area alone.

**What I did:**
1. Filled missing basement values with 0 (no basement = 0 sq ft)
2. Created `TotalSF = GrLivArea + TotalBsmtSF`
3. Plotted TotalSF vs SalePrice
4. Calculated Pearson correlation

**Why I chose what I chose:**
- **Fill NA with 0, not drop:** Missing basement means no basement. Dropping would lose houses that are perfectly valid.
- **Pearson correlation:** Both variables continuous, relationship roughly linear.
- **Compare to Q2:** Same dataset, same method — fair comparison.

**What I found:**
| Feature | Correlation |
|---------|-------------|
| GrLivArea alone | 0.709 |
| TotalSF (engineered) | 0.779 |
| Improvement | +0.070 |

**What this means:**
- Feature engineering works — creating `TotalSF` captured information that `GrLivArea` missed
- Basement area does matter for price, even if less valuable per sq ft than above-ground space
- Not a huge jump (0.07), but meaningful — and shows why data scientists don't just trust raw columns

**Code I learned:**
| What I wanted | How I wrote it |
|---------------|----------------|
| Fill missing values | `df['col'].fillna(0, inplace=True)` |
| Create new column | `df['NewCol'] = df['A'] + df['B']` |
| Compare correlations | Look side by side |

**What I'd try next:**
Weight basement area less than above-ground (e.g., `TotalSF_weighted = GrLivArea + 0.5 * TotalBsmtSF`) and see if correlation improves further.

## Q7 — Correlation heatmap of all numeric features

**Goal:** Identify which numeric features have the strongest correlation with SalePrice, and visualize all correlations in a heatmap.

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
numeric_df = numeric_df.drop(columns=['Id','log_saleprice'])
correlation_matrix = numeric_df.corr()
plt.figure(figsize=(10,8))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm',
            fmt='.2f', linewidths=0.5, center=0, square=True)
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.savefig("../visuals/correlation_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("=== Correlation Analysis Results ===\n")
print("Correlation with SalePrice:")
print(correlation_matrix['SalePrice'].sort_values(ascending=False))
print("\n")

corr_values = correlation_matrix.unstack().sort_values(ascending=False)
corr_values = corr_values[corr_values < 0.999]
print("Top Correlations (excluding 1.0):")
print(corr_values.head(5))
print("\n")

print("Top Negative Correlations:")
print(corr_values.tail(5))

### Q7 — Conclusion

**What I was trying to figure out:**
Which numeric features have the strongest relationship with SalePrice, and how do features relate to each other?

**Why I chose what I chose:**
- **Filtered to numeric columns only:** Categorical variables need different handling (frequency encoding, one-hot)
- **Dropped 'Id':** Random identifier has no predictive value
- **Dropped 'log_saleprice':** It's just a transformed version of SalePrice — including it would artificially inflate correlations and be circular logic
- **Heatmap instead of just numbers:** Heatmap reveals patterns — clusters of correlated features show multicollinearity
- **No annotation (annot=False):** Too many features would be unreadable with numbers

**What I found:**

**Top 5 features correlated with SalePrice:**
| Rank | Feature | Correlation |
|------|---------|-------------|
| 1 | OverallQual | 0.791 |
| 2 | TotalSF (engineered) | 0.779 |
| 3 | GrLivArea | 0.709 |
| 4 | GarageCars | 0.640 |
| 5 | GarageArea | 0.623 |

**What surprised me:**
- `OverallQual` beat my engineered `TotalSF` — quality matters more than size
- `GarageCars` and `GarageArea` have 0.882 correlation with each other — they're measuring almost the same thing (redundant)
- `KitchenAbvGr` has negative correlation (-0.136) — more kitchens = lower price (probably because extra kitchens indicate converted multi-family units or unusual layouts)

**Multicollinearity detected (high correlation between features):**
| Feature Pair | Correlation | What it means |
|--------------|-------------|----------------|
| GarageCars ↔ GarageArea | 0.882 | Pick one, drop the other |
| GrLivArea ↔ TotalSF | 0.880 | TotalSF contains GrLivArea, so keep only TotalSF |
| YearBuilt ↔ GarageYrBlt | 0.826 | Newer houses have newer garages |

**What this taught me:**
- Heatmaps are great for feature selection before ML
- High correlation between features (multicollinearity) means I should drop redundant features
- Filtering to |r| > 0.4 for the target helps focus on what actually predicts price

**Code I learned:**
| What I wanted | How I wrote it |
|---------------|----------------|
| Select numeric columns | `df.select_dtypes(include=[np.number])` |
| Drop specific columns | `df.drop(columns=['col1', 'col2'])` |
| Full correlation matrix | `df.corr()` |
| Correlations with target | `df.corr()['SalePrice'].sort_values()` |

**What I'd do next for ML:**
Recommended feature set (based on this analysis):
- `OverallQual` (strongest predictor)
- `TotalSF` (engineered, captures all living space)
- `GarageCars` (keep, drop GarageArea)
- `Neighborhood` (categorical — needs encoding)
- `YearBuilt` or `YearRemodAdd` (pick one)

**Features to drop due to multicollinearity:**
- `GarageArea` (covered by GarageCars)
- `GrLivArea` (covered by TotalSF)
- `log_saleprice` (already removed — was circular)